In [1]:
import requests
import pandas as pd
import joblib
import numpy as np

In [29]:
def get_live_prediction(lat, lon, model_path, pca_path, scaler_path):
    # --- 1. SETUP ---
    headers = {'User-Agent': '(WPI_Research, ctwalsh@wpi.edu)'}
    
    data = None

    model = joblib.load(model_path)
    pca = joblib.load(pca_path)
    scaler = joblib.load(scaler_path)
    
    # Hardcoded Maxima from your TRAINING set for the Inversion
    T_MAX_TRAIN = 85.0 
    H_MAX_TRAIN = 95.0 
    D_MAX_TRAIN = 75.0 

    try:
        # STEP 1: Find the nearest station for the given Lat/Lon
        point_url = f"https://api.weather.gov/points/{lat},{lon}"
        point_res = requests.get(point_url, headers=headers).json()
    
        # Get the observations URL for that specific location
        # Note: For locations outside the US (like Churchill), NWS won't work.
        # For Churchill, you'd need the Environment Canada API.
        obs_url = point_res['properties']['observationStations']
        stations = requests.get(obs_url, headers=headers).json()
        primary_station = stations['features'][0]['id'] # Get the first nearby station
    
        # STEP 2: Fetch observations from THAT station
        final_obs_url = f"{primary_station}/observations?limit=500"
        response = requests.get(final_obs_url, headers=headers)

        if response.status_code == 200:
            data = response.json()
        else:
            print(f"NWS Server returned error code: {response.status_code}")
            return None

        # 3. VERIFY KEY EXISTENCE
        if data is None or 'features' not in data:
            print("NWS Response received but 'features' key is missing.")
            return None

        # 4. PROCESS RECORDS
        records = []
        for p in data['features']:
            prop = p.get('properties', {})
            
            # Use .get() to safely pull nested values
            temp_val = prop.get('temperature', {}).get('value')
            dew_val = prop.get('dewpoint', {}).get('value')
            hum_val = prop.get('relativeHumidity', {}).get('value')
            precip_val = prop.get('precipitationLastHour', {}).get('value') or 0
            wind_val = prop.get('windSpeed', {}).get('value') or 0
            
            if temp_val is not None and dew_val is not None:
                records.append({
                    'timestamp': prop.get('timestamp'),
                    'temp': (temp_val * 9/5) + 32, # Fahrenheit conversion
                    'humidity': hum_val,
                    'dew_point': (dew_val * 9/5) + 32,
                    'precip': precip_val,
                    'wind_speed': wind_val
                })

    except Exception as e:
        print(f"An error occurred: {e}")
        return None
    
    if not records:
        raise Exception("No valid weather records found in the last 500 observations.")
    
    df_raw = pd.DataFrame(records).dropna()
    df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
    

    # Raw Features
    df_raw['Temp_Change_Velocity'] = df_raw['temp'].diff()
    
    df_raw['In_Risk_Window'] = (
        (df_raw['temp'].between(15, 35)) & (df_raw['dew_point'].between(10, 30))
    ).astype(int)
    
    # 1. Resample to get the baseline weekly means
    df_weekly = df_raw.resample('W', on='timestamp').mean().sort_index()

    # 2. RENAME IMMEDIATELY (This fixes the 'not in index' error)
    df_weekly = df_weekly.rename(columns={
    'temp': 'avg_temp_prior_week',
    'humidity': 'avg_humidity_prior_week',
    'dew_point': 'avg_dew_point_prior_week',
    'precip': 'avg_precip_prior_week',
    'wind_speed': 'avg_wind_speed_prior_week'
    })

    # 3. NOW calculate RSI using the new names
    df_weekly['temp_stress'] = T_MAX_TRAIN - df_weekly['avg_temp_prior_week']
    df_weekly['hum_stress'] = H_MAX_TRAIN - df_weekly['avg_humidity_prior_week']
    df_weekly['dew_stress'] = D_MAX_TRAIN - df_weekly['avg_dew_point_prior_week']
    
    df_weekly['RSI'] = (df_weekly['temp_stress'] + df_weekly['hum_stress'] + df_weekly['dew_stress']) / 3


    # RSI Sustained & Volatility
    df_weekly['RSI_Sustained'] = df_weekly['RSI'].rolling(window=4, min_periods=1).mean()
    df_weekly['RSI_Volatility'] = df_weekly['RSI'].rolling(window=3, min_periods=1).std().fillna(0)
    


    # Step A: Variables for PCA (Must be in this EXACT order)
    pca_input_vars = [
    'RSI', 
    'RSI_Sustained', 
    'RSI_Volatility', 
    'avg_precip_prior_week', 
    'avg_wind_speed_prior_week'
    ]

    # 'bfill' (backfill) takes the first valid value and copies it to the NaN spots
    df_weekly = df_weekly.ffill().bfill().fillna(0)

     # --- 4. THE HYBRID ASSEMBLY ---
    # Grab the most recent week of data
    latest = df_weekly.iloc[[-1]]

    # 3. Final Safety Check: If there is STILL a NaN, replace with 0
    if latest.isnull().values.any():
        latest = latest.fillna(0)

    # Scale and Transform
    scaled_pca_input = scaler.transform(latest[pca_input_vars])
    pca_output = pca.transform(scaled_pca_input) # Results in PC1, PC2, etc.

    # Step B: Raw Variables for the final concatenation
    raw_input_vars = ['Temp_Change_Velocity', 'In_Risk_Window']
    raw_vals = latest[raw_input_vars].values

    # --- ADD THIS MISSING LINE ---
    X_inference = np.hstack([pca_output, raw_vals])
    # -----------------------------

    # --- Step C: Re-label the data before prediction ---
    # Create the list of names in the EXACT order the model expects
    # Replace 'PC1', 'PC2', etc., with however many components your PCA has
    pca_cols = [f'PC{i+1}' for i in range(pca_output.shape[1])]
    final_feature_names = pca_cols + ['Temp_Change_Velocity', 'In_Risk_Window']

    # Convert the numpy stack back into a labeled DataFrame
    X_final_df = pd.DataFrame(X_inference, columns=final_feature_names)

    

    # --- TEMPORARY TEST CASE ---
    # Let's simulate a 'Cold Shock' and put us in the Risk Window
    #X_final_df.at[0, 'Temp_Change_Velocity'] = -15.0  # Significant drop
    #X_final_df.at[0, 'In_Risk_Window'] = 1.0          # Inside 15-35°F / 10-30 Dew
    #X_final_df.at[0, 'PC1'] = 2.5                     # High RSI stress

    print("Final Feature Vector for Model:")
    print(X_final_df)

    # --- 5. PREDICTION ---
    # Now the model will see the feature names it expects
    log_pred = model.predict(X_final_df)
    prob = np.expm1(log_pred)
    
    return prob[0]

# Usage
try:
    current_prediction = get_live_prediction(64.84, -147.72, 'gb_model.pkl', 'pca_transformer.pkl', 'data_scaler.pkl')
    # Define the bins you used in your EDA/Training
    breaks = [0, 4.67898e-05, 0.000154293, 0.0002945734, 0.0008872]
    labels = ['Low', 'Moderate', 'High', 'Critical']

    current_category = pd.cut([current_prediction], bins=breaks, labels=labels)[0]
    print(f"Live Prediction Score: {current_prediction:.6f}")
    print(f"Risk Category: {current_category}")
except Exception as e:
    print(f"Inference Failed: {e}")

Final Feature Vector for Model:
        PC1       PC2       PC3       PC4  Temp_Change_Velocity  \
0  1.646194 -2.126285 -0.952519  0.979986             -0.003607   

   In_Risk_Window  
0           0.052  
Live Prediction Score: 0.000054
Risk Category: Moderate
